In [ ]:
from pyspark.sql import SparkSession
# 1. Khởi tạo Spark Session kết nối đến Master chung của cụm
spark = SparkSession.builder \
    .appName("MetroPT3_Machine_Learning") \
    .master("spark://10.125.222.18:7077") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()
# 2. Đường dẫn nạp dữ liệu sạch đã lưu trên HDFS
HDFS_ML = "hdfs://10.125.222.18:9000/user/bigdata/cleaned/metropt3_clean_for_sql"
df = spark.read.parquet(HDFS_ML)
# 4. Kiểm tra cấu trúc dữ liệu để bắt đầu làm Classification
df.printSchema()
print(f"Tổng số bản ghi: {df.count()}")
df.createOrReplaceTempView('sensor')

In [ ]:
# Q3: PHÂN TÍCH CƯỜNG ĐỘ VẬN HÀNH THEO TỪNG NGÀY TRONG TUẦN
spark.sql('''
    CREATE OR REPLACE TEMP VIEW daily_activity AS
    SELECT
        date,
        timestamp,
        COMP,
        Oil_temperature,
        Motor_current,
        CASE DAYOFWEEK(date)
            WHEN 2 THEN '02. Thứ 2'
            WHEN 3 THEN '03. Thứ 3'
            WHEN 4 THEN '04. Thứ 4'
            WHEN 5 THEN '05. Thứ 5'
            WHEN 6 THEN '06. Thứ 6'
            WHEN 7 THEN '07. Thứ 7'
            WHEN 1 THEN '01. Chủ Nhật'
        END AS ngay_trong_tuan,
        -- Khởi động = chuyển từ trạng thái nghỉ sang chạy (1 -> 0)
        CASE
            WHEN COMP = 0
             AND LAG(COMP,1,1)
                 OVER ( PARTITION BY date ORDER BY timestamp) = 1
            THEN 1 ELSE 0
        END AS la_khoi_dong
    FROM sensor''')
q3_weekly_profile = spark.sql('''
    SELECT
        ngay_trong_tuan,
        ROUND(
            SUM(la_khoi_dong) /
            COUNT(DISTINCT date)
        ,2) AS trung_binh_lan_khoi_dong,
        ROUND(
            AVG(CASE WHEN COMP = 0 THEN 1 ELSE 0 END) * 100
        ,1) AS ty_le_thoi_gian_chay_pct,
        ROUND(
            AVG(CASE WHEN COMP = 0 THEN Oil_temperature END)
        ,2) AS nhiet_do_dau_tb,
        ROUND(
            AVG(CASE WHEN COMP = 0 THEN Motor_current END)
        ,2) AS dong_dien_tb
    FROM daily_activity
    GROUP BY ngay_trong_tuan
    ORDER BY ngay_trong_tuan
''')
print("\n========== EXECUTION PLAN ==========\n")
q3_weekly_profile.explain(True)
print("\n--- CƯỜNG ĐỘ VẬN HÀNH THEO TỪNG NGÀY TRONG TUẦN ---")
df_q3_weekly = q3_weekly_profile.toPandas()
try:
    display(df_q3_weekly)
except NameError:
    print(df_q3_weekly.to_string(index=False))